In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score



In [ ]:
df = pd.read_csv(r"C:\KLH\2026-2027\Subjects\Y25-ML\practical\Placement-Prediction-System\data\placement_predict_50k Dataset.csv")

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
# ------------------------------------------------------------
# Select placed students
# ------------------------------------------------------------
placed_df = df[df["Salary Package"] > 0].copy()

print("Placed students:", len(placed_df))



In [ ]:
# ------------------------------------------------------------
# Simple Linear Regression
# X = CGPA
# y = Salary Package
# ------------------------------------------------------------
X = placed_df[["CGPA"]]
y = placed_df["Salary Package"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

In [ ]:
#Standardize CGPA for Gradient Descent


scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Original CGPA:")
print(X_train.head())

print("\nStandardized CGPA:")
print(X_train_scaled[:5])

In [ ]:
#Batch Gradient Descent from scratch

# Convert to NumPy arrays
X_gd = X_train_scaled.flatten()
y_gd = y_train.to_numpy()

m = len(X_gd)

# Initialize parameters
theta_0 = 0.0
theta_1 = 0.0

learning_rate = 0.01
epochs = 1000

loss_history = []

# ------------------------------------------------------------
# Batch Gradient Descent
# ------------------------------------------------------------
for epoch in range(epochs):

    # Prediction
    y_pred = theta_0 + theta_1 * X_gd
    print(" theta_0:", theta_0)
    print(" theta_1:", theta_1)

    # Error
    error = y_pred - y_gd

    # MSE
    mse = np.mean(error ** 2)
    loss_history.append(mse)

    # Gradients
    gradient_theta_0 = (2 / m) * np.sum(error)
    gradient_theta_1 = (2 / m) * np.sum(error * X_gd)

    # Parameter update
    theta_0 -= learning_rate * gradient_theta_0
    theta_1 -= learning_rate * gradient_theta_1

print("Final theta_0:", theta_0)
print("Final theta_1:", theta_1)
print("Final training MSE:", loss_history[-1])

In [ ]:
#MSE Curve loss

plt.figure(figsize=(8, 5))

plt.plot(
    range(1, epochs + 1),
    loss_history,
    linewidth=2
)

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Batch Gradient Descent: MSE Loss vs Epoch")
plt.grid(True, alpha=0.3)

plt.show()

In [ ]:
X_test_gd = X_test_scaled.flatten()

y_test_pred_gd = theta_0 + theta_1 * X_test_gd

gd_rmse = np.sqrt(
    mean_squared_error(y_test, y_test_pred_gd)
)

gd_mae = mean_absolute_error(
    y_test,
    y_test_pred_gd
)

gd_r2 = r2_score(
    y_test,
    y_test_pred_gd
)

print("Batch GD RMSE:", gd_rmse)
print("Batch GD MAE :", gd_mae)
print("Batch GD R²  :", gd_r2)

In [ ]:
#Compare with sklearn LinearRegression
sk_model = LinearRegression()

sk_model.fit(
    X_train_scaled,
    y_train
)

y_test_pred_sk = sk_model.predict(
    X_test_scaled
)

sk_rmse = np.sqrt(
    mean_squared_error(y_test, y_test_pred_sk)
)

sk_mae = mean_absolute_error(
    y_test,
    y_test_pred_sk
)

sk_r2 = r2_score(
    y_test,
    y_test_pred_sk
)

print("Sklearn RMSE:", sk_rmse)
print("Sklearn MAE :", sk_mae)
print("Sklearn R²  :", sk_r2)

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Batch Gradient Descent",
        "sklearn LinearRegression"
    ],
    "RMSE": [
        gd_rmse,
        sk_rmse
    ],
    "MAE": [
        gd_mae,
        sk_mae
    ],
    "R²": [
        gd_r2,
        sk_r2
    ]
})

display(comparison)

## Using "Mini-Batch Gradient Descent" 

In [ ]:
# ------------------------------------------------------------
# Mini-Batch Gradient Descent
# ------------------------------------------------------------

X_mb = X_train_scaled.flatten()
y_mb = y_train.to_numpy()

batch_size = 32
epochs_mb = 1000
learning_rate_mb = 0.01

theta_0_mb = 0.0
theta_1_mb = 0.0

mini_batch_loss = []

rng = np.random.default_rng(42)

n_samples = len(X_mb)

for epoch in range(epochs_mb):

    # Shuffle training data
    indices = rng.permutation(n_samples)

    X_shuffled = X_mb[indices]
    y_shuffled = y_mb[indices]

    # Process mini-batches
    for start in range(0, n_samples, batch_size):

        end = start + batch_size

        X_batch = X_shuffled[start:end]
        y_batch = y_shuffled[start:end]

        batch_n = len(X_batch)

        # Prediction
        y_batch_pred = (
            theta_0_mb +
            theta_1_mb * X_batch
        )

        # Error
        error = y_batch_pred - y_batch

        # Gradients
        grad_0 = (2 / batch_n) * np.sum(error)

        grad_1 = (
            (2 / batch_n) *
            np.sum(error * X_batch)
        )

        # Update
        theta_0_mb -= learning_rate_mb * grad_0
        theta_1_mb -= learning_rate_mb * grad_1

    # Calculate full training loss after each epoch
    y_epoch_pred = (
        theta_0_mb +
        theta_1_mb * X_mb
    )

    epoch_mse = np.mean(
        (y_epoch_pred - y_mb) ** 2
    )

    mini_batch_loss.append(epoch_mse)

print("Final theta_0:", theta_0_mb)
print("Final theta_1:", theta_1_mb)
print("Final MSE:", mini_batch_loss[-1])

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    loss_history,
    label="Batch GD",
    linewidth=2
)

plt.plot(
    mini_batch_loss,
    label="Mini-Batch GD (batch size=32)",
    linewidth=2
)

plt.xlabel("Epoch")
plt.ylabel("Training MSE")
plt.title("Batch GD vs Mini-Batch GD")
plt.legend()
plt.grid(True, alpha=0.3)

plt.show()